# Vector Calculus

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 02.08 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/09_vector_calculus.ipynb)

---

## 🎯 Learning Objective

Learn divergence, curl, and the Laplacian as tools for describing vector fields, and preview Green's and Stokes' theorems.

---

## 📐 Theory

A **vector field** $\mathbf{F}(x,y) = (P(x,y), Q(x,y))$ assigns a vector to every point in
space — think of it as describing a flow, like wind velocity at every point on a map. Vector
calculus gives you scalar and vector tools to describe how such a flow behaves locally.

### Divergence: is the flow expanding or contracting?

The **divergence** of $\mathbf{F}$ is a scalar field:

$$\text{div}\,\mathbf{F} = \nabla\cdot\mathbf{F} = \frac{\partial P}{\partial x} + \frac{\partial Q}{\partial y}$$

At a point, $\text{div}\,\mathbf{F} > 0$ means the flow is a net "source" there (more flow
leaves a tiny neighborhood than enters — like a point emitting fluid); $\text{div}\,\mathbf{F}<0$
means a net "sink." The notation $\nabla\cdot\mathbf{F}$ treats $\nabla=(\partial/\partial x,
\partial/\partial y)$ as a vector of operators and takes its formal dot product with
$\mathbf{F}$ — a mnemonic that also explains the gradient's notation from `02.03`
($\nabla f$ is this same "vector of operators" applied to a scalar).

### Curl: is the flow rotating?

In 2D, the (scalar) curl measures local rotation:

$$\text{curl}\,\mathbf{F} = \frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}$$

A positive curl means the flow locally rotates counterclockwise around that point (imagine
dropping a tiny paddle wheel into the flow and watching which way it spins); zero curl
everywhere means the field is **irrotational** — informally, it has no local "swirl," which
turns out to be exactly the condition under which $\mathbf{F}$ can be written as the gradient
of some scalar function ($\mathbf{F} = \nabla f$ for some $f$, called a **conservative** field).

### The Laplacian

Applying divergence to a gradient gives the **Laplacian**, a single scalar operator combining
all pure second derivatives:

$$\nabla^2 f = \nabla\cdot(\nabla f) = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2}$$

This is the trace of the Hessian from `02.06` — summing curvature along each axis rather than
reporting the full matrix. $\nabla^2 f = 0$ (Laplace's equation) characterizes functions with
no local "bumps or dips" relative to their neighborhood average — a smoothness condition that
appears throughout physics and, as we'll see below, in the mathematics of diffusion.

### Green's and Stokes' theorems (previewed)

**Green's theorem** relates a line integral around a closed curve $C$ to a double integral of
curl over the region $R$ it encloses: $\oint_C \mathbf{F}\cdot d\mathbf{r} = \iint_R
\text{curl}\,\mathbf{F}\,dA$. **Stokes' theorem** is the 3D generalization, relating a surface's
boundary circulation to the curl integrated over the surface. Both say the same conceptual
thing: total rotation around a boundary equals accumulated local rotation throughout the
interior — a "local information sums to global behavior" idea structurally similar to the
Fundamental Theorem of Calculus from `02.07`. We won't need the full theorems computationally
in this notebook, but the intuition — boundary behavior reflects an integral of interior
behavior — reappears in the score-based generative models below.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting three contrasting vector fields side by side — pure outward flow (positive
divergence, zero curl), pure rotation (zero divergence, nonzero curl), and a gradient field
(irrotational by construction) — makes divergence and curl visually distinguishable at a glance.

In [ ]:
# Three canonical vector fields: pure source (div>0, curl=0), pure rotation (div=0, curl>0),
# and a gradient field (always curl=0, by construction -- the "conservative field" fact)
fields = {
    "Source: F=(x,y)\ndiv=2, curl=0": lambda x, y: (x, y),
    "Rotation: F=(-y,x)\ndiv=0, curl=2": lambda x, y: (-y, x),
    "Gradient of x^2+y^2: F=(2x,2y)\ndiv=4, curl=0 (irrotational)": lambda x, y: (2*x, 2*y),
}

x, y = np.meshgrid(np.linspace(-2, 2, 12), np.linspace(-2, 2, 12))
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, field) in zip(axes, fields.items()):
    U, V = field(x, y)
    ax.quiver(x, y, U, V, color="#4C72B0")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print("Source field: arrows point outward everywhere -- positive divergence, no net rotation.")
print("Rotation field: arrows circle counterclockwise -- zero divergence, positive curl.")
print("Gradient field: curl is ALWAYS zero for a gradient field -- this is why conservative")
print("fields (those expressible as grad(f)) are exactly the irrotational ones.")

## 🐍 Implementation from Scratch

We compute divergence, curl, and the Laplacian numerically from their definitions on sampled
grids, and confirm that a gradient field always has zero curl.

In [ ]:
def divergence_2d(P_grid, Q_grid, x_1d, y_1d):
    """div F = dP/dx + dQ/dy, computed on a sampled grid. np.gradient(arr, y_1d, x_1d) on a
    meshgrid('xy')-style array returns [d/dy, d/dx] since axis 0 varies with y."""
    _, dP_dx = np.gradient(P_grid, y_1d, x_1d)
    dQ_dy, _ = np.gradient(Q_grid, y_1d, x_1d)
    return dP_dx + dQ_dy

def curl_2d(P_grid, Q_grid, x_1d, y_1d):
    """(scalar) curl F = dQ/dx - dP/dy."""
    dP_dy, _ = np.gradient(P_grid, y_1d, x_1d)
    _, dQ_dx = np.gradient(Q_grid, y_1d, x_1d)
    return dQ_dx - dP_dy

def laplacian_2d(f_grid, x_1d, y_1d):
    """laplacian f = d^2f/dx^2 + d^2f/dy^2 -- divergence of the gradient."""
    df_dy, df_dx = np.gradient(f_grid, y_1d, x_1d)
    _, d2f_dx2 = np.gradient(df_dx, y_1d, x_1d)
    d2f_dy2, _ = np.gradient(df_dy, y_1d, x_1d)
    return d2f_dx2 + d2f_dy2

x_1d = np.linspace(-2, 2, 60)
y_1d = np.linspace(-2, 2, 60)
Xg, Yg = np.meshgrid(x_1d, y_1d)

# Source field F=(x,y): analytic divergence=2, curl=0 everywhere
div_source = divergence_2d(Xg, Yg, x_1d, y_1d)
curl_source = curl_2d(Xg, Yg, x_1d, y_1d)
print(f"Source field F=(x,y):     mean div = {div_source.mean():.4f} (expect 2), mean curl = {curl_source.mean():.6f} (expect 0)")

# Rotation field F=(-y,x): analytic divergence=0, curl=2 everywhere
div_rot = divergence_2d(-Yg, Xg, x_1d, y_1d)
curl_rot = curl_2d(-Yg, Xg, x_1d, y_1d)
print(f"Rotation field F=(-y,x):  mean div = {div_rot.mean():.4f} (expect 0), mean curl = {curl_rot.mean():.4f} (expect 2)")

# Gradient field F = grad(x^2 + y^2) = (2x, 2y): curl should be ~0 EVERYWHERE (conservative field fact)
curl_grad_field = curl_2d(2 * Xg, 2 * Yg, x_1d, y_1d)
print(f"Gradient field grad(x^2+y^2): max |curl| = {np.max(np.abs(curl_grad_field)):.2e} (expect ~0, up to numerical noise)")

# Laplacian of f(x,y) = x^2 + y^2: analytic answer is 2+2=4 everywhere
lap = laplacian_2d(Xg**2 + Yg**2, x_1d, y_1d)
print(f"\nLaplacian of x^2+y^2 (interior points): {lap[10:-10, 10:-10].mean():.4f}  (expect 4)")

## 🤖 Why This Matters for AI

Diffusion models (fully derived in `09.06`) learn a **score function**
$\mathbf{s}(\mathbf{x}) = \nabla_{\mathbf{x}} \log p(\mathbf{x})$ — a vector field pointing
toward regions of higher data density at every point in space, exactly the kind of object this
notebook studies. Since a score function is a *gradient* field by construction, it's always
irrotational (curl-free), a fact this notebook lets you verify directly rather than take on
faith. **Langevin dynamics** generates new samples by literally simulating a particle flowing
through this vector field, nudged by the score plus a little noise:
$\mathbf{x}_{t+1} = \mathbf{x}_t + \eta\,\mathbf{s}(\mathbf{x}_t) + \sqrt{2\eta}\,\boldsymbol{\epsilon}_t$.
Below, we compute the score field of a simple 2D Gaussian mixture, confirm it's curl-free, and
run Langevin dynamics to sample from it — watching particles flow "uphill" along the score
field toward the two modes.

In [ ]:
# Score function of a 2-component Gaussian mixture, verified curl-free, then used to
# drive Langevin dynamics sampling -- the mechanism behind score-based diffusion models.
means = [np.array([-2.0, 0.0]), np.array([2.0, 1.0])]
sigma = 0.8

def score(x):
    # grad_x log p(x) for an equal-weight Gaussian mixture: a responsibility-weighted
    # average of each component's own (simple, Gaussian) score
    comps = [np.exp(-np.sum((x - m) ** 2) / (2 * sigma**2)) for m in means]
    total = sum(comps)
    resp = [c / total for c in comps]
    return sum(r * (-(x - m) / sigma**2) for r, m in zip(resp, means))

# Confirm the score field is curl-free (it's a gradient field, so it MUST be, by this
# notebook's own theorem) -- reuse curl_2d on a grid of score vectors
x_1d_s = np.linspace(-5, 5, 60)
y_1d_s = np.linspace(-3, 4, 60)
Xs, Ys = np.meshgrid(x_1d_s, y_1d_s)
Ps = np.zeros_like(Xs); Qs = np.zeros_like(Ys)
for i in range(Xs.shape[0]):
    for j in range(Xs.shape[1]):
        g = score(np.array([Xs[i, j], Ys[i, j]]))
        Ps[i, j], Qs[i, j] = g[0], g[1]
curl_score = curl_2d(Ps, Qs, x_1d_s, y_1d_s)
score_scale = max(np.abs(Ps).max(), np.abs(Qs).max())
print(f"Score field max |curl| (interior): {np.max(np.abs(curl_score[5:-5, 5:-5])):.4f}")
print(f"...compared to the score field's own scale (~{score_scale:.2f}): small, and it shrinks")
print(f"further on a finer grid -- the residual is finite-difference discretization error,")
print(f"not real curl (a true gradient field's curl is EXACTLY zero analytically).")

# Langevin dynamics: let particles flow uphill along the score field, plus noise, and
# watch them settle near the two true mixture modes without ever being TOLD where the modes are
rng = np.random.default_rng(0)
n_particles, eta, n_steps = 150, 0.05, 300
particles = rng.normal(scale=3, size=(n_particles, 2))
trajectory = [particles.copy()]
for _ in range(n_steps):
    grads = np.array([score(p) for p in particles])
    noise = rng.normal(size=particles.shape)
    particles = particles + eta * grads + np.sqrt(2 * eta) * noise
    trajectory.append(particles.copy())

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].streamplot(Xs, Ys, Ps, Qs, color="#4C72B0", density=1.2)
for m in means:
    axes[0].plot(*m, 'r*', markersize=15)
axes[0].set_title("Score field (a gradient field -> curl-free)\nstars = true mixture modes")

axes[1].scatter(trajectory[0][:, 0], trajectory[0][:, 1], alpha=0.4, color="#DD8452", label="start")
axes[1].scatter(particles[:, 0], particles[:, 1], alpha=0.6, color="#55A868", label="after Langevin dynamics")
for m in means:
    axes[1].plot(*m, 'r*', markersize=15)
axes[1].set_title("Langevin dynamics: particles flow along\nthe score field toward the true modes")
axes[1].legend()
plt.tight_layout()
plt.show()

## 🏋️ Exercises

### Warm-up
1. For $\mathbf{F}(x,y) = (3x^2, -2y)$, compute $\text{div}\,\mathbf{F}$ and
   $\text{curl}\,\mathbf{F}$ by hand. Verify with `divergence_2d`/`curl_2d`.

### Practice
2. For $f(x,y) = \sin(x)\cos(y)$, compute the Laplacian $\nabla^2 f$ by hand, then verify
   numerically with `laplacian_2d`. At which points in $[-\pi,\pi]^2$ is $\nabla^2 f = 0$?

### Challenge
3. Add a third mixture component to the `means` list in the AI section (with its own mean),
   re-run Langevin dynamics, and confirm particles now settle near all three modes. Then
   increase `eta` by 5x and describe what goes wrong — connect your observation to why
   diffusion model samplers use carefully scheduled, typically small step sizes rather than
   one large jump (a preview of learning rate schedules in Module `04.06`).

---

## 📚 Further Reading
- Song & Ermon, ["Generative Modeling by Estimating Gradients of the Data Distribution"](https://arxiv.org/abs/1907.05600) (score-based generative models)
- Strang, *Calculus*, Ch. 15 (Vector Calculus)

---

*Next notebook: [The Calculus of Backpropagation](10_calculus_of_backpropagation.ipynb)*